In [1]:
# conda activate anndata

import numpy as np
import pandas as pd
import anndata as ad

Here I create pseduobulk data per donor and cell type to perform DE analysis for each cell type

In [ ]:
import os

os.chdir("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/pseudobulk_test/yao_2021/MOp")

In [ ]:
adata = ad.read_h5ad("data/yao_2021_MOp_STAR_model/yao_2021_MOp_STAR_gene_counts_scVI.h5ad")

In [ ]:
# I want to find DE genes for each cell type. The best way to do this is by pseudobulking cell types by donor.

# First let's see how the makeup of cell types break down by donor:

pd.DataFrame(adata.obs.groupby("donor_id")["subclass_label"].value_counts().groupby(level=0).head(1))

/tmp/ipykernel_150413/4090677257.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pd.DataFrame(adata.obs.groupby("external_donor_name_label")["cell_type"].value_counts().groupby(level=0).head(1))
/tmp/ipykernel_150413/4090677257.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pd.DataFrame(adata.obs.groupby("external_donor_name_label")["cell_type"].value_counts().groupby(level=0).head(1))


,,count
external_donor_name_label,cell_type,
328806,L4/5 intratelencephalic projecting glutamatergic neuron of the primary motor cortex,122
329689,L4/5 intratelencephalic projecting glutamatergic neuron of the primary motor cortex,71
334440,L4/5 intratelencephalic projecting glutamatergic neuron of the primary motor cortex,94
335069,L6 corticothalamic-projecting glutamatergic cortical neuron,124
337056,L6 corticothalamic-projecting glutamatergic cortical neuron,142
337058,L6 corticothalamic-projecting glutamatergic cortical neuron,126
337059,L6 corticothalamic-projecting glutamatergic cortical neuron,102
340649,L4/5 intratelencephalic projecting glutamatergic neuron of the primary motor cortex,121
341906,L6 corticothalamic-projecting glutamatergic cortical neuron,140


In [ ]:
cell_meta = adata.obs
cells_per_class = cell_meta['subclass_label'].value_counts()
cells_per_class[cells_per_class > 10].index.values

['L6 corticothalamic-projecting glutamatergic c..., 'L4/5 intratelencephalic projecting glutamater..., 'L2/3-6 intratelencephalic projecting glutamat..., 'L6b glutamatergic cortical neuron', 'L6 intratelencephalic projecting glutamatergi..., ..., 'sst GABAergic cortical interneuron', 'pvalb GABAergic cortical interneuron', 'L5 extratelencephalic projecting glutamatergi..., 'lamp5 GABAergic cortical interneuron', 'sncg GABAergic cortical interneuron']
Length: 12
Categories (19, object): ['endothelial cell', 'astrocyte', 'oligodendrocyte', 'pyramidal neuron', ..., 'L6 intratelencephalic projecting glutamatergi..., 'vascular leptomeningeal cell (Mmus)', 'meis2 expressing cortical GABAergic cell', 'sst chodl GABAergic cortical interneuron']

In [ ]:
# Remove cell types with < 6 cells:

min_cells = 6

cells_to_keep = cells_per_class[cells_per_class >= min_cells].index.values.astype("str").tolist()

adata = adata[adata.obs['subclass_label'].isin(cells_to_keep)].copy()

In [ ]:
adata.shape

In [16]:
adata.is_view

False

In [ ]:
df_list = []
meta_list = []

for ctype in np.unique(adata.obs['subclass_label']):
    print(f"Starting {ctype}...")
    
    adata_subset = adata[adata.obs['subclass_label'] == ctype].copy()
    
    X = adata_subset.raw.X
    
    df = pd.DataFrame.sparse.from_spmatrix(
        X, 
        index=adata_subset.obs_names,
        columns=adata.raw.var_names
    )
    df_bulked = df.groupby(adata_subset.obs['donor_id']).sum()
    
    meta_list.append(pd.DataFrame({
        'Cell_type': ctype, 
        'Donor': df_bulked.index.astype(str).values
    }))
    
    df_bulked.index = ctype + "_" + df_bulked.index.astype(str)
    df_list.append(df_bulked)

In [20]:
df_all = pd.concat(df_list, axis=0)
meta = pd.concat(meta_list, axis=0).reset_index(drop=True)

In [ ]:
df_all.shape

In [22]:
meta['Sample_ID'] = df_all.index.values
df_all.index.name = None

In [ ]:
# Save
df_all.T.to_csv("data/yao_2021_MOp_STAR_donor_subclass_label_pseudobulk.csv")
meta.to_csv("data/yao_2021_MOp_STAR_donor_subclass_label_pseudobulk_sampleinfo.csv", index=False)